# 02. Analyze

**Input:** `../data/clean.pkl`, `../data/raw.pkl`.
**Does:** the accountability gradient across actor types, the observed severity split by representation, an
ordered logit fit with court-clustered standard errors, and a scikit-learn severity classifier with
permutation importance and a confusion matrix.
**Output:** five figures in `../output/`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier
np.random.seed(20)
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})

clean = pd.read_pickle("../data/clean.pkl")
raw   = pd.read_pickle("../data/raw.pkl")

## (A) Accountability gradient (all actor types, full database)

In [ ]:
def lever(o):
    if pd.isna(o): return "None recorded"
    s = str(o).lower()
    if any(k in s for k in ["bar referr","referral","suspen","disqualif"," dq","pro hac","censure",
        "grievance","disciplinary","revocation","revoked","admonish"]): return "Professional discipline"
    if any(k in s for k in ["monetary","fine","costs order","cost order","adverse cost","attorney fee",
        "fees and cost","$","damages","disgorge","penalty"]): return "Monetary"
    if any(k in s for k in ["struck","stricken","strike","dismiss","vacat","remand","reversed","set aside",
        "quash","withdrew","withdrawn","annul","retract","corrected","refile","amend","waived","default"]):
        return "Case/ruling remedy"
    if any(k in s for k in ["warning","caution","reprimand","rebuke","flag"]): return "Warning only"
    return "None recorded"

def actor_full(p):
    if pd.isna(p): return None
    if "Judge" in p: return "Judge"
    if p=="Pro Se Litigant": return "Pro se litigant"
    if p in {"Lawyer","Governement Lawyer","Prosecutor","Federal Defender"}: return "Lawyer (counseled)"
    return None

raw["actor"]=raw["Party(ies)"].apply(actor_full); raw["lever"]=raw["Outcome"].apply(lever)
order=["Lawyer (counseled)","Pro se litigant","Judge"]
levels=["Professional discipline","Monetary","Case/ruling remedy","Warning only","None recorded"]
g=raw[raw["actor"].isin(order)]
tab=(g.groupby("actor")["lever"].value_counts(normalize=True).unstack()
       .reindex(index=order,columns=levels).fillna(0)*100)
n=g["actor"].value_counts().reindex(order)
print(tab.round(1).to_string())

cols={"Professional discipline":"#7b1f2b","Monetary":"#c0504d","Case/ruling remedy":"#e0a458",
      "Warning only":"#8ab6d6","None recorded":"#cccccc"}
fig,ax=plt.subplots(figsize=(9,4.8)); left=np.zeros(len(order))
for lv in levels:
    v=tab[lv].values; ax.barh(range(len(order)),v,left=left,color=cols[lv],label=lv,edgecolor="white"); left+=v
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{a}\n(n={int(n[a])})" for a in order]); ax.invert_yaxis()
ax.set_xlim(0,100); ax.set_xlabel("% of that actor's cases")
ax.set_title("Who bears the consequence for an AI hallucination?")
ax.legend(ncol=3,fontsize=8.5,frameon=False,loc="upper center",bbox_to_anchor=(0.5,-0.15))
plt.tight_layout(); plt.savefig("../output/fig_accountability_gradient.png",dpi=200,bbox_inches="tight"); plt.close()
print("saved ../output/fig_accountability_gradient.png")

## (B) Observed severity by representation

In [ ]:
d0=clean[clean["actor"].isin(["Pro se","Counseled"])]
dist=(d0.groupby("actor")["severity"].value_counts(normalize=True).unstack()
        .reindex(index=["Pro se","Counseled"],columns=range(5)).fillna(0)*100)
labs=["0 None","1 Warning","2 Procedural","3 Monetary","4 Prof/\nterminal"]; xg=np.arange(5); w=0.38
fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar(xg-w/2,dist.loc["Pro se"].values,w,label=f"Pro se (n={int((clean.actor=='Pro se').sum())})",color="#4C72B0")
ax.bar(xg+w/2,dist.loc["Counseled"].values,w,label=f"Counseled (n={int((clean.actor=='Counseled').sum())})",color="#c0504d")
ax.set_xticks(xg); ax.set_xticklabels(labs,fontsize=9); ax.set_ylabel("% of cases")
ax.set_title("Sanction-severity distribution by representation"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("../output/fig_severity_dist.png",dpi=200); plt.close()
print("saved ../output/fig_severity_dist.png")

## (C) Ordered logit (proportional odds), court-clustered SEs

In [ ]:
d = clean[clean["actor"].isin(["Pro se","Counseled"])].dropna(subset=["year"]).copy()
d["pro_se"]=(d["actor"]=="Pro se").astype(int); d["year_c"]=d["year"]-2025
fd=pd.get_dummies(d["field"],prefix="f",drop_first=True).astype(float).reset_index(drop=True)
Xcols=["pro_se","federal","year_c"]+list(fd.columns)
X=pd.concat([d[["pro_se","federal","year_c"]].astype(float).reset_index(drop=True),fd],axis=1).values
y=d["severity"].values.astype(int); court=d["Court"].values; p=X.shape[1]

def Sg_(z): return 1/(1+np.exp(-z))
def cuts(th):
    c=np.empty(4); c[0]=th[0]
    for k in range(1,4): c[k]=c[k-1]+np.exp(th[k])
    return c
def ll_obs(par):
    b=par[:p]; c=cuts(par[p:p+4]); eta=X@b
    lo=np.where(y==0,-np.inf,np.take(c,np.clip(y-1,0,3)))
    hi=np.where(y==4, np.inf,np.take(c,np.clip(y,0,3)))
    return np.log(np.clip(np.where(np.isinf(hi),1,Sg_(hi-eta))-np.where(np.isinf(lo),0,Sg_(lo-eta)),1e-12,1))
negll=lambda par:-ll_obs(par).sum()
r=minimize(negll,np.r_[np.zeros(p),[-1,0,0,0]],method="Nelder-Mead",options={"maxiter":40000,"fatol":1e-8,"xatol":1e-8})
r=minimize(negll,r.x,method="BFGS",options={"maxiter":5000}); beta=r.x[:p]

def pg(par,eps=1e-5):
    G=np.zeros((len(y),len(par)))
    for j in range(len(par)):
        a=par.copy();a[j]+=eps;b=par.copy();b[j]-=eps;G[:,j]=(ll_obs(a)-ll_obs(b))/(2*eps)
    return G
def hess(par,eps=1e-4):
    nP=len(par);H=np.zeros((nP,nP))
    for i in range(nP):
        for j in range(i,nP):
            a=par.copy();a[i]+=eps;a[j]+=eps;b=par.copy();b[i]+=eps;b[j]-=eps
            c=par.copy();c[i]-=eps;c[j]+=eps;e=par.copy();e[i]-=eps;e[j]-=eps
            H[i,j]=H[j,i]=(negll(a)-negll(b)-negll(c)+negll(e))/(4*eps*eps)
    return H
Sm=pg(r.x); Hi=np.linalg.pinv(hess(r.x)); meat=np.zeros((len(r.x),)*2)
for gg in np.unique(court):
    u=Sm[court==gg].sum(0); meat+=np.outer(u,u)
se=np.sqrt(np.diag(Hi@meat@Hi))[:p]
res=pd.DataFrame({"term":Xcols,"OR":np.exp(beta),"lo":np.exp(beta-1.96*se),
                  "hi":np.exp(beta+1.96*se),"p":2*(1-norm.cdf(np.abs(beta/se)))})
print(res.round(3).to_string(index=False))

show=["pro_se","federal","year_c"]+[c for c in Xcols if c.startswith("f_")]
sub=res.set_index("term").loc[show]; yy=np.arange(len(show))[::-1]
fig,ax=plt.subplots(figsize=(8,5))
ax.errorbar(sub["OR"],yy,xerr=[sub["OR"]-sub["lo"],sub["hi"]-sub["OR"]],fmt="o",color="#2f4b7c",capsize=3)
ax.axvline(1,color="#999",ls="--",lw=1); ax.set_yticks(yy); ax.set_yticklabels(show,fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Odds ratio (higher severity), log scale")
ax.set_title("Ordered-logit odds ratios, 95% CI (court-clustered)")
plt.tight_layout(); plt.savefig("../output/fig_forest.png",dpi=200); plt.close()
print("saved ../output/fig_forest.png")

## (D) Supervised ML: predicting severity (scikit-learn)

In [ ]:
feat=pd.concat([d[["pro_se","federal","year","tool_named"]].reset_index(drop=True),
                pd.get_dummies(d["field"],prefix="f").astype(int).reset_index(drop=True)],axis=1)
Xtr,Xte,ytr,yte=train_test_split(feat,y,test_size=0.25,stratify=y,random_state=20)
base=DummyClassifier(strategy="most_frequent").fit(Xtr,ytr)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000)).fit(Xtr,ytr)
pr=clf.predict(Xte)
print("baseline acc:",round((base.predict(Xte)==yte).mean(),3),
      "| model acc:",round((pr==yte).mean(),3),
      "| macro-F1:",round(f1_score(yte,pr,average='macro'),3))

pi=permutation_importance(clf,Xte,yte,n_repeats=30,random_state=20,scoring="f1_macro")
imp=pd.Series(pi.importances_mean,index=feat.columns).sort_values()
fig,ax=plt.subplots(figsize=(7,4.2)); ax.barh(range(len(imp)),imp.values,color="#2f4b7c")
ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index,fontsize=9)
ax.set_xlabel("Permutation importance (macro-F1 drop)"); ax.set_title("What predicts sanction severity?")
plt.tight_layout(); plt.savefig("../output/fig_ml_importance.png",dpi=200); plt.close()

cm=confusion_matrix(yte,pr,normalize="true")
labs2=["None","Warning","Procedural","Monetary","Prof/term"]
fig,ax=plt.subplots(figsize=(5.5,4.8)); im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(labs2,rotation=40,ha="right"); ax.set_yticklabels(labs2)
for i in range(5):
    for j in range(5): ax.text(j,i,f"{cm[i,j]:.2f}",ha="center",va="center",fontsize=8,color="white" if cm[i,j]>.5 else "#333")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (row-normalized)")
plt.colorbar(im,fraction=0.046); plt.tight_layout(); plt.savefig("../output/fig_ml_confusion.png",dpi=200); plt.close()
print("saved ../output/fig_ml_importance.png and ../output/fig_ml_confusion.png")